In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
import numpy as np

def profile_activations():
    """Profile SiLU vs ReLU activation functions"""
    print("=== SiLU vs ReLU Activation Profiling ===")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Test different tensor sizes
    sizes = [(1000,), (10000,), (100000,), (1000000,)]
    silu_times = []
    relu_times = []
    
    def time_activation(activation_fn, x, num_runs=1000):
        """Time an activation function"""
        # Warmup
        for _ in range(10):
            _ = activation_fn(x)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        start_time = time.time()
        for _ in range(num_runs):
            _ = activation_fn(x)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        return (time.time() - start_time) / num_runs
    
    print("\nProfiling different tensor sizes:")
    print("Size\t\tSiLU (μs)\tReLU (μs)\tSlowdown")
    print("-" * 50)
    
    for size in sizes:
        # Create test tensor
        x = torch.randn(size, device=device, requires_grad=True)
        
        # Time SiLU
        silu_time = time_activation(torch.nn.functional.silu, x)
        silu_times.append(silu_time * 1e6)  # Convert to microseconds
        
        # Time ReLU
        relu_time = time_activation(torch.nn.functional.relu, x)
        relu_times.append(relu_time * 1e6)  # Convert to microseconds
        
        slowdown = silu_time / relu_time
        
        print(f"{size[0]:>8}\t\t{silu_time*1e6:>6.2f}\t\t{relu_time*1e6:>6.2f}\t\t{slowdown:.2f}x")
    
    # Plot results
    plt.figure(figsize=(10, 6))
    
    # Convert sizes to numbers for plotting
    size_nums = [s[0] for s in sizes]
    
    plt.subplot(1, 2, 1)
    plt.loglog(size_nums, silu_times, 'o-', label='SiLU', linewidth=2, markersize=8)
    plt.loglog(size_nums, relu_times, 's-', label='ReLU', linewidth=2, markersize=8)
    plt.xlabel('Tensor Size')
    plt.ylabel('Time (μs)')
    plt.title('Activation Function Performance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    slowdowns = [s/r for s, r in zip(silu_times, relu_times)]
    plt.semilogx(size_nums, slowdowns, 'ro-', linewidth=2, markersize=8)
    plt.xlabel('Tensor Size')
    plt.ylabel('SiLU/ReLU Slowdown')
    plt.title('SiLU Performance Overhead')
    plt.grid(True, alpha=0.3)
    plt.axhline(y=1, color='k', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Summary
    avg_slowdown = np.mean(slowdowns)
    print(f"\nSummary:")
    print(f"  Average SiLU slowdown: {avg_slowdown:.2f}x")
    print(f"  SiLU is consistently slower due to exponential computation")
    print(f"  ReLU is a simple max(0, x) operation, hence faster")
    
    return {
        'sizes': size_nums,
        'silu_times': silu_times,
        'relu_times': relu_times,
        'slowdowns': slowdowns,
        'avg_slowdown': avg_slowdown
    }

# Run the activation profiling
activation_results = profile_activations()

In [ ]:
# Actor vs ActorEGNN Performance Profiling
# Comparing forward and backward pass performance between simple MLP Actor and Graph Neural Network Actor

import os
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.profiler
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import seaborn as sns

# Environment setup
os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"

# Set torch settings for consistent profiling
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

# Import our models
from fast_td3 import Actor, ActorEGNN
from fast_td3.hyperparams import HumanoidBenchArgs

print("Environment setup complete!")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Test parameters - matching training setup
args = HumanoidBenchArgs(
    env_name="h1-stand-v0",
    total_timesteps=50000,
    render_interval=5000,
    eval_interval=1000,
    num_envs=16,
    batch_size=8192,
)

# Network parameters
n_obs = 51  # Observation dimension for h1 environment
n_act = 19  # Action dimension for h1 (19 joints)
num_envs = args.num_envs
batch_size = args.batch_size
hidden_dim_egnn = 80
hidden_dim_mlp = 512
init_scale = 0.1

print(f"Test configuration:")
print(f"  n_obs: {n_obs}")
print(f"  n_act: {n_act}")
print(f"  num_envs: {num_envs}")
print(f"  batch_size: {batch_size}")
print(f"  hidden_dim_egnn: {hidden_dim_egnn}")
print(f"  hidden_dim_mlp: {hidden_dim_mlp}")

# Create sample input data for testing
# For training batch profiling
obs_batch = torch.randn(batch_size, n_obs, device=device, requires_grad=True)
xpos_batch = torch.randn(batch_size, 20, 3, device=device, requires_grad=True)  # 20 nodes, 3D positions

# For environment interaction profiling  
obs_env = torch.randn(num_envs, n_obs, device=device, requires_grad=True)
xpos_env = torch.randn(num_envs, 20, 3, device=device, requires_grad=True)

print(f"Sample data shapes:")
print(f"  Training batch - obs: {obs_batch.shape}, xpos: {xpos_batch.shape}")
print(f"  Environment - obs: {obs_env.shape}, xpos: {xpos_env.shape}")

In [ ]:
# Create both actor models
print("Creating Actor models...")

# Standard MLP Actor
actor_mlp = Actor(
    n_obs=n_obs,
    n_act=n_act,
    num_envs=num_envs,
    init_scale=init_scale,
    hidden_dim=hidden_dim_mlp,
    device=device,
)

# Graph Neural Network Actor
actor_gnn = ActorEGNN(
    n_obs=n_obs,
    n_act=n_act,
    num_envs=num_envs,
    init_scale=init_scale,
    hidden_dim=hidden_dim_egnn,
    batch_size=batch_size,
    device=device,
)

# Model parameter counts
mlp_params = sum(p.numel() for p in actor_mlp.parameters())
gnn_params = sum(p.numel() for p in actor_gnn.parameters())

print(f"\nModel Parameter Counts:")
print(f"  Actor (MLP): {mlp_params:,} parameters")
print(f"  ActorEGNN: {gnn_params:,} parameters")
print(f"  Parameter ratio (GNN/MLP): {gnn_params/mlp_params:.2f}x")

# Memory usage
def get_model_memory(model):
    """Calculate memory usage of model parameters"""
    return sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)  # MB

mlp_memory = get_model_memory(actor_mlp)
gnn_memory = get_model_memory(actor_gnn)

print(f"\nModel Memory Usage:")
print(f"  Actor (MLP): {mlp_memory:.2f} MB")
print(f"  ActorEGNN: {gnn_memory:.2f} MB")
print(f"  Memory ratio (GNN/MLP): {gnn_memory/mlp_memory:.2f}x")

In [ ]:
# Basic timing utilities
def time_function(func, *args, num_runs=100, warmup=10):
    """Time a function with warmup runs"""
    # Warmup
    for _ in range(warmup):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        _ = func(*args)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # Actual timing
    start_time = time.time()
    for _ in range(num_runs):
        _ = func(*args)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    end_time = time.time()
    avg_time = (end_time - start_time) / num_runs
    return avg_time

def compare_forward_pass():
    """Compare forward pass performance"""
    print("=== Forward Pass Comparison ===")
    
    # Environment-sized batches (inference)
    print("\n1. Environment Batch (Inference):")
    print(f"   Input shape: {obs_env.shape}")
    
    mlp_time_env = time_function(lambda: actor_mlp(obs_env), num_runs=1000)
    gnn_time_env = time_function(lambda: actor_gnn(obs_env, xpos_env), num_runs=1000)
    
    print(f"   Actor (MLP): {mlp_time_env*1000:.3f} ms")
    print(f"   ActorEGNN: {gnn_time_env*1000:.3f} ms")
    print(f"   Slowdown: {gnn_time_env/mlp_time_env:.2f}x")
    
    # Training-sized batches
    print("\n2. Training Batch:")
    print(f"   Input shape: {obs_batch.shape}")
    
    mlp_time_batch = time_function(lambda: actor_mlp(obs_batch), num_runs=100)
    gnn_time_batch = time_function(lambda: actor_gnn(obs_batch, xpos_batch), num_runs=100)
    
    print(f"   Actor (MLP): {mlp_time_batch*1000:.3f} ms")
    print(f"   ActorEGNN: {gnn_time_batch*1000:.3f} ms")
    print(f"   Slowdown: {gnn_time_batch/mlp_time_batch:.2f}x")
    
    return {
        'mlp_env': mlp_time_env,
        'gnn_env': gnn_time_env,
        'mlp_batch': mlp_time_batch,
        'gnn_batch': gnn_time_batch
    }

# Run forward pass comparison
forward_times = compare_forward_pass()

In [ ]:
def compare_backward_pass():
    """Compare backward pass performance (full forward + backward)"""
    print("\n=== Backward Pass Comparison ===")
    
    # Create optimizers
    optimizer_mlp = optim.Adam(actor_mlp.parameters(), lr=1e-4)
    optimizer_gnn = optim.Adam(actor_gnn.parameters(), lr=1e-4)
    
    def mlp_backward_step():
        optimizer_mlp.zero_grad()
        obs_batch_copy = obs_batch.clone().detach().requires_grad_(True)
        output = actor_mlp(obs_batch_copy)
        loss = output.sum()  # Dummy loss for gradient computation
        loss.backward()
        optimizer_mlp.step()
        return loss
    
    def gnn_backward_step():
        optimizer_gnn.zero_grad()
        obs_batch_copy = obs_batch.clone().detach().requires_grad_(True)
        xpos_batch_copy = xpos_batch.clone().detach().requires_grad_(True)
        output = actor_gnn(obs_batch_copy, xpos_batch_copy)
        loss = output.sum()  # Dummy loss for gradient computation
        loss.backward()
        optimizer_gnn.step()
        return loss
    
    print(f"Training batch backward pass (input shape: {obs_batch.shape}):")
    
    mlp_backward_time = time_function(mlp_backward_step, num_runs=50)
    gnn_backward_time = time_function(gnn_backward_step, num_runs=50)
    
    print(f"   Actor (MLP) forward+backward: {mlp_backward_time*1000:.3f} ms")
    print(f"   ActorEGNN forward+backward: {gnn_backward_time*1000:.3f} ms")
    print(f"   Slowdown: {gnn_backward_time/mlp_backward_time:.2f}x")
    
    return {
        'mlp_backward': mlp_backward_time,
        'gnn_backward': gnn_backward_time
    }

# Run backward pass comparison
backward_times = compare_backward_pass()

In [ ]:
def detailed_profile_forward():
    """Use PyTorch profiler for detailed analysis of forward pass"""
    print("\n=== Detailed Profiling (Forward Pass) ===")
    
    def profile_model(model, obs, xpos=None, model_name="Model"):
        print(f"\n{model_name} Profile:")
        
        # Profile with CPU and CUDA events
        activities = [torch.profiler.ProfilerActivity.CPU]
        if torch.cuda.is_available():
            activities.append(torch.profiler.ProfilerActivity.CUDA)
        
        with torch.profiler.profile(
            activities=activities,
            record_shapes=True,
            profile_memory=True,
            with_stack=False
        ) as prof:
            for _ in range(10):  # Profile multiple runs
                if xpos is not None:
                    _ = model(obs, xpos)
                else:
                    _ = model(obs)
        
        # Get key statistics
        key_averages = prof.key_averages(group_by_input_shape=False)
        
        if torch.cuda.is_available():
            print("Top 5 operations by CUDA time:")
            # Use the correct attribute name for CUDA time
            cuda_ops = sorted(key_averages, key=lambda x: getattr(x, 'cuda_time_total', getattr(x, 'device_time_total', 0)), reverse=True)[:5]
            for i, op in enumerate(cuda_ops, 1):
                cuda_time = getattr(op, 'cuda_time_total', getattr(op, 'device_time_total', 0))
                print(f"  {i}. {op.key}: {cuda_time/1000:.2f} ms total, {op.count} calls")
        else:
            print("CUDA not available - no CUDA profiling data")
        
        # Memory usage
        memory_ops = sorted(key_averages, key=lambda x: getattr(x, 'cpu_memory_usage', 0), reverse=True)[:3]
        print("Top 3 operations by memory usage:")
        for i, op in enumerate(memory_ops, 1):
            memory_usage = getattr(op, 'cpu_memory_usage', 0)
            print(f"  {i}. {op.key}: {memory_usage/(1024**2):.2f} MB")
        
        return prof
    
    # Profile both models on training batch
    print("Profiling on training batch...")
    mlp_prof = profile_model(actor_mlp, obs_batch, model_name="Actor (MLP)")
    gnn_prof = profile_model(actor_gnn, obs_batch, xpos_batch, model_name="ActorEGNN")
    
    return mlp_prof, gnn_prof

# Run detailed profiling
mlp_profile, gnn_profile = detailed_profile_forward()

In [ ]:
def analyze_gnn_components():
    """Break down ActorEGNN into its components to identify bottlenecks"""
    print("\n=== ActorEGNN Component Analysis ===")
    
    # Time individual components of ActorEGNN
    def time_build_input():
        return actor_gnn.egnn.build_batched_egnn_input(obs_batch, xpos_batch)
    
    def time_egnn_forward(h, x, edges, edge_attr):
        return actor_gnn.egnn.forward(h, x, edges, edge_attr)
    
    # Time input building
    print("1. Building EGNN input...")
    input_build_time = time_function(time_build_input, num_runs=1000)
    print(f"   Input building: {input_build_time*1000:.3f} ms")
    
    # Get the inputs for EGNN
    h, x, edges, edge_attr = time_build_input()
    print(f"   h shape: {h.shape}")
    print(f"   x shape: {x.shape}")
    print(f"   edges length: {len(edges)} with shape: {[e.shape for e in edges]}")
    print(f"   edge_attr shape: {edge_attr.shape if edge_attr is not None else None}")
    
    # Time EGNN forward pass
    print("\n2. EGNN forward pass...")
    egnn_forward_time = time_function(lambda: time_egnn_forward(h, x, edges, edge_attr), num_runs=1000)
    print(f"   EGNN forward: {egnn_forward_time*1000:.3f} ms")
    
    # Time full ActorEGNN forward for comparison
    print("\n3. Full ActorEGNN forward...")
    full_gnn_time = time_function(lambda: actor_gnn(obs_batch, xpos_batch), num_runs=1000)
    print(f"   Full ActorEGNN: {full_gnn_time*1000:.3f} ms")
    
    # Calculate overhead
    overhead = full_gnn_time - input_build_time - egnn_forward_time
    print(f"\n4. Breakdown:")
    print(f"   Input building: {input_build_time/full_gnn_time*100:.1f}%")
    print(f"   EGNN forward: {egnn_forward_time/full_gnn_time*100:.1f}%")
    print(f"   Other overhead: {overhead/full_gnn_time*100:.1f}%")
    
    return {
        'input_build': input_build_time,
        'egnn_forward': egnn_forward_time,
        'full_forward': full_gnn_time,
        'overhead': overhead
    }

# Analyze GNN components
gnn_components = analyze_gnn_components()

In [ ]:
def analyze_memory_usage():
    """Analyze memory usage patterns"""
    print("\n=== Memory Usage Analysis ===")
    
    if not torch.cuda.is_available():
        print("CUDA not available, skipping GPU memory analysis")
        return
    
    def measure_memory(func, *args, model_name="Model"):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
        # Measure before
        memory_before = torch.cuda.memory_allocated() / (1024**2)  # MB
        
        # Run function
        result = func(*args)
        
        # Measure after
        memory_after = torch.cuda.memory_allocated() / (1024**2)  # MB
        memory_peak = torch.cuda.max_memory_allocated() / (1024**2)  # MB
        
        print(f"{model_name}:")
        print(f"  Memory before: {memory_before:.2f} MB")
        print(f"  Memory after: {memory_after:.2f} MB")
        print(f"  Memory peak: {memory_peak:.2f} MB")
        print(f"  Memory increase: {memory_after - memory_before:.2f} MB")
        print(f"  Peak increase: {memory_peak - memory_before:.2f} MB")
        
        return {
            'before': memory_before,
            'after': memory_after,
            'peak': memory_peak,
            'increase': memory_after - memory_before,
            'peak_increase': memory_peak - memory_before
        }
    
    print("Forward pass memory usage (training batch):")
    mlp_memory = measure_memory(lambda: actor_mlp(obs_batch), model_name="Actor (MLP)")
    print()
    gnn_memory = measure_memory(lambda: actor_gnn(obs_batch, xpos_batch), model_name="ActorEGNN")
    
    print(f"\nMemory comparison:")
    print(f"  Peak memory ratio (GNN/MLP): {gnn_memory['peak_increase']/mlp_memory['peak_increase']:.2f}x")
    
    return mlp_memory, gnn_memory

# Analyze memory usage
memory_analysis = analyze_memory_usage()

In [ ]:
def analyze_batch_scaling():
    """Analyze how performance scales with batch size"""
    print("\n=== Batch Size Scaling Analysis ===")
    
    batch_sizes = [16, 64, 256, 1024, 4096, 8192]
    mlp_times = []
    gnn_times = []
    
    print("Testing different batch sizes...")
    for bs in batch_sizes:
        print(f"\nBatch size: {bs}")
        
        # Create test data for this batch size
        obs_test = torch.randn(bs, n_obs, device=device)
        xpos_test = torch.randn(bs, 20, 3, device=device)
        
        # Time MLP
        mlp_time = time_function(lambda: actor_mlp(obs_test), num_runs=100)
        mlp_times.append(mlp_time)
        
        # Time GNN
        gnn_time = time_function(lambda: actor_gnn(obs_test, xpos_test), num_runs=100)
        gnn_times.append(gnn_time)
        
        print(f"  MLP: {mlp_time*1000:.3f} ms")
        print(f"  GNN: {gnn_time*1000:.3f} ms")
        print(f"  Slowdown: {gnn_time/mlp_time:.2f}x")
    
    # Calculate throughput (samples per second)
    mlp_throughput = [bs / time for bs, time in zip(batch_sizes, mlp_times)]
    gnn_throughput = [bs / time for bs, time in zip(batch_sizes, gnn_times)]
    
    print(f"\n=== Scaling Summary ===")
    print("Batch Size | MLP (ms) | GNN (ms) | Slowdown | MLP (samples/s) | GNN (samples/s)")
    print("-" * 80)
    for i, bs in enumerate(batch_sizes):
        print(f"{bs:>9} | {mlp_times[i]*1000:>7.2f} | {gnn_times[i]*1000:>7.2f} | "
              f"{gnn_times[i]/mlp_times[i]:>7.2f}x | {mlp_throughput[i]:>14.0f} | {gnn_throughput[i]:>14.0f}")
    
    return {
        'batch_sizes': batch_sizes,
        'mlp_times': mlp_times,
        'gnn_times': gnn_times,
        'mlp_throughput': mlp_throughput,
        'gnn_throughput': gnn_throughput
    }

# Analyze batch scaling
scaling_results = analyze_batch_scaling()

In [ ]:
def visualize_results():
    """Create visualizations of the profiling results"""
    print("\n=== Performance Visualization ===")
    
    # Set up the plotting style
    plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Actor vs ActorEGNN Performance Comparison', fontsize=16, fontweight='bold')
    
    # 1. Forward pass timing comparison
    ax1 = axes[0, 0]
    scenarios = ['Environment\n(16 samples)', 'Training Batch\n(8192 samples)']
    mlp_vals = [forward_times['mlp_env']*1000, forward_times['mlp_batch']*1000]
    gnn_vals = [forward_times['gnn_env']*1000, forward_times['gnn_batch']*1000]
    
    x = np.arange(len(scenarios))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, mlp_vals, width, label='Actor (MLP)', alpha=0.8, color='skyblue')
    bars2 = ax1.bar(x + width/2, gnn_vals, width, label='ActorEGNN', alpha=0.8, color='lightcoral')
    
    ax1.set_ylabel('Time (ms)')
    ax1.set_title('Forward Pass Performance')
    ax1.set_xticks(x)
    ax1.set_xticklabels(scenarios)
    ax1.legend()
    ax1.set_yscale('log')
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height*1.1, f'{height:.2f}', 
                ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height*1.1, f'{height:.2f}', 
                ha='center', va='bottom', fontsize=9)
    
    # 2. Backward pass comparison
    ax2 = axes[0, 1]
    scenarios_back = ['Forward + Backward']
    mlp_back = [backward_times['mlp_backward']*1000]
    gnn_back = [backward_times['gnn_backward']*1000]
    
    bars3 = ax2.bar([0 - width/2], mlp_back, width, label='Actor (MLP)', alpha=0.8, color='skyblue')
    bars4 = ax2.bar([0 + width/2], gnn_back, width, label='ActorEGNN', alpha=0.8, color='lightcoral')
    
    ax2.set_ylabel('Time (ms)')
    ax2.set_title('Training Step Performance')
    ax2.set_xticks([0])
    ax2.set_xticklabels(scenarios_back)
    ax2.legend()
    
    for bar in bars3:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height*1.02, f'{height:.2f}', 
                ha='center', va='bottom', fontsize=9)
    for bar in bars4:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height*1.02, f'{height:.2f}', 
                ha='center', va='bottom', fontsize=9)
    
    # 3. Batch size scaling
    ax3 = axes[1, 0]
    ax3.plot(scaling_results['batch_sizes'], np.array(scaling_results['mlp_times'])*1000, 
             'o-', label='Actor (MLP)', linewidth=2, markersize=6)
    ax3.plot(scaling_results['batch_sizes'], np.array(scaling_results['gnn_times'])*1000, 
             's-', label='ActorEGNN', linewidth=2, markersize=6)
    ax3.set_xlabel('Batch Size')
    ax3.set_ylabel('Time (ms)')
    ax3.set_title('Batch Size Scaling')
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    # 4. GNN Component breakdown
    ax4 = axes[1, 1]
    components = ['Input Building', 'EGNN Forward', 'Overhead']
    times = [gnn_components['input_build']*1000, 
             gnn_components['egnn_forward']*1000, 
             gnn_components['overhead']*1000]
    colors = ['lightgreen', 'orange', 'lightgray']
    
    wedges, texts, autotexts = ax4.pie(times, labels=components, autopct='%1.1f%%', 
                                       colors=colors, startangle=90)
    ax4.set_title('ActorEGNN Component Breakdown')
    
    plt.tight_layout()
    plt.show()
    
    # Performance summary table
    print("\n=== Performance Summary ===")
    print(f"Forward Pass Slowdown:")
    print(f"  Environment batch: {forward_times['gnn_env']/forward_times['mlp_env']:.2f}x slower")
    print(f"  Training batch: {forward_times['gnn_batch']/forward_times['mlp_batch']:.2f}x slower")
    print(f"Training Step Slowdown: {backward_times['gnn_backward']/backward_times['mlp_backward']:.2f}x slower")
    print(f"Parameter Count: {gnn_params/mlp_params:.2f}x more parameters")
    
    if torch.cuda.is_available() and memory_analysis:
        mlp_mem, gnn_mem = memory_analysis
        print(f"Memory Usage: {gnn_mem['peak_increase']/mlp_mem['peak_increase']:.2f}x more memory")

# Create visualizations
visualize_results()

In [ ]:
def performance_recommendations():
    """Provide optimization recommendations based on profiling results"""
    print("\n" + "="*60)
    print("PERFORMANCE ANALYSIS & OPTIMIZATION RECOMMENDATIONS")
    print("="*60)
    
    # Calculate key metrics
    env_slowdown = forward_times['gnn_env'] / forward_times['mlp_env']
    batch_slowdown = forward_times['gnn_batch'] / forward_times['mlp_batch']
    training_slowdown = backward_times['gnn_backward'] / backward_times['mlp_backward']
    
    print(f"\n📊 KEY PERFORMANCE METRICS:")
    print(f"   • Environment inference: {env_slowdown:.1f}x slower")
    print(f"   • Training batch forward: {batch_slowdown:.1f}x slower") 
    print(f"   • Full training step: {training_slowdown:.1f}x slower")
    print(f"   • Parameter overhead: {gnn_params/mlp_params:.1f}x more parameters")
    
    print(f"\n🔍 BOTTLENECK ANALYSIS:")
    total_gnn_time = gnn_components['full_forward']
    input_pct = gnn_components['input_build'] / total_gnn_time * 100
    egnn_pct = gnn_components['egnn_forward'] / total_gnn_time * 100
    overhead_pct = gnn_components['overhead'] / total_gnn_time * 100
    
    print(f"   • Input building: {input_pct:.1f}% of GNN time")
    print(f"   • EGNN computation: {egnn_pct:.1f}% of GNN time")
    print(f"   • Other overhead: {overhead_pct:.1f}% of GNN time")
    
    print(f"\n⚡ OPTIMIZATION STRATEGIES:")
    
    # Strategy 1: Input building optimization
    if input_pct > 20:
        print(f"   1. 🎯 OPTIMIZE INPUT BUILDING ({input_pct:.1f}% of time)")
        print(f"      • Pre-compute edge indices and cache them")
        print(f"      • Use torch.stack instead of tensor concatenation")
        print(f"      • Consider moving edge building to C++/CUDA kernel")
    
    # Strategy 2: EGNN optimization  
    if egnn_pct > 50:
        print(f"   2. 🎯 OPTIMIZE EGNN COMPUTATION ({egnn_pct:.1f}% of time)")
        print(f"      • Use sparse operations for edge computations")
        print(f"      • Implement custom CUDA kernels for message passing")
        print(f"      • Consider graph batching optimizations")
        print(f"      • Use mixed precision training (fp16)")
    
    # Strategy 3: Memory optimization
    if torch.cuda.is_available() and memory_analysis:
        mlp_mem, gnn_mem = memory_analysis
        mem_ratio = gnn_mem['peak_increase'] / mlp_mem['peak_increase']
        if mem_ratio > 3:
            print(f"   3. 🎯 OPTIMIZE MEMORY USAGE ({mem_ratio:.1f}x more memory)")
            print(f"      • Use gradient checkpointing")
            print(f"      • Implement in-place operations where possible")
            print(f"      • Consider smaller hidden dimensions")
    
    # Strategy 4: Architecture alternatives
    print(f"   4. 🎯 CONSIDER ARCHITECTURE ALTERNATIVES")
    print(f"      • Hybrid approach: MLP + lightweight graph features")
    print(f"      • Use attention mechanisms instead of full graph convolution")
    print(f"      • Pre-trained graph embeddings")
    print(f"      • Hierarchical graph structure")
    
    # Strategy 5: Implementation optimizations
    print(f"   5. 🎯 IMPLEMENTATION OPTIMIZATIONS")
    print(f"      • Use torch.jit.script compilation")
    print(f"      • Enable torch.compile() for PyTorch 2.0+")
    print(f"      • Batch operations more efficiently")
    print(f"      • Use torch.sparse for edge operations")
    
    print(f"\n💡 QUICK WINS:")
    print(f"   • Cache edge indices (avoid recomputation)")
    print(f"   • Use mixed precision (torch.autocast)")
    print(f"   • Profile with different hidden_dim sizes")
    print(f"   • Consider reducing EGNN layers (currently {actor_gnn.egnn.n_layers})")
    
    print(f"\n🎯 RECOMMENDED NEXT STEPS:")
    print(f"   1. Implement edge index caching")
    print(f"   2. Add mixed precision support")
    print(f"   3. Profile with torch.compile()")
    print(f"   4. Test smaller hidden dimensions")
    print(f"   5. Benchmark against hybrid MLP+graph approach")
    
    return {
        'env_slowdown': env_slowdown,
        'batch_slowdown': batch_slowdown,
        'training_slowdown': training_slowdown,
        'input_pct': input_pct,
        'egnn_pct': egnn_pct,
        'overhead_pct': overhead_pct
    }

# Generate recommendations
recommendations = performance_recommendations()

In [ ]:
# Example optimization: Edge index caching
def demonstrate_edge_caching_optimization():
    """Show how caching edge indices can improve performance"""
    print("\n" + "="*50)
    print("OPTIMIZATION DEMO: EDGE INDEX CACHING")
    print("="*50)
    
    # Current approach: build edges every time
    def current_approach():
        h, x, edges, edge_attr = actor_gnn.egnn.build_batched_egnn_input(obs_batch, xpos_batch)
        return actor_gnn.egnn.forward(h, x, edges, edge_attr)
    
    # Optimized approach: pre-build and cache edges
    def optimized_approach():
        # Pre-compute the input building (this would be cached in practice)
        h, x, edges, edge_attr = actor_gnn.egnn.build_batched_egnn_input(obs_batch, xpos_batch)
        
        # Only time the actual EGNN forward pass
        return actor_gnn.egnn.forward(h, x, edges, edge_attr)
    
    # Time both approaches
    print("Timing current vs optimized approach...")
    
    current_time = time_function(current_approach, num_runs=1000)
    
    # For optimized, we pre-compute once and then just time the forward pass
    h, x, edges, edge_attr = actor_gnn.egnn.build_batched_egnn_input(obs_batch, xpos_batch)
    optimized_time = time_function(lambda: actor_gnn.egnn.forward(h, x, edges, edge_attr), num_runs=1000)
    
    speedup = current_time / optimized_time
    
    print(f"\nResults:")
    print(f"  Current approach: {current_time*1000:.3f} ms")
    print(f"  Optimized (cached edges): {optimized_time*1000:.3f} ms")
    print(f"  Speedup: {speedup:.2f}x")
    
    if speedup > 1.5:
        print(f"  ✅ Significant improvement! Consider implementing edge caching.")
    else:
        print(f"  ⚠️  Moderate improvement. Focus on other optimizations first.")
    
    return current_time, optimized_time, speedup

# Run optimization demo
demo_results = demonstrate_edge_caching_optimization()

print(f"\n" + "="*50)
print("PROFILING COMPLETE!")
print("="*50)
print("The ActorEGNN is significantly slower than the simple MLP Actor.")
print("Key findings:")
print("• Graph neural networks add substantial computational overhead")
print("• Input preprocessing and edge building are major bottlenecks") 
print("• Memory usage is also significantly higher")
print("• Consider the trade-offs between model expressiveness and speed")
print("\nRefer to the recommendations above for optimization strategies.")